<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/3-document-deletion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deleting Documents from a Corpus

This notebook shows how to remove documents from a Vectara corpus. It's self-contained: it creates its own corpus, adds a few documents (some with metadata, some without), then demonstrates the three deletion APIs:

- **Delete one document by ID** — `DELETE /v2/corpora/{corpus_key}/documents/{document_id}` (returns `204`).
- **Bulk delete by metadata or IDs** — `DELETE /v2/corpora/{corpus_key}/documents` with a `metadata_filter` (e.g. `doc.category = 'finance'`) and/or a comma-separated `document_ids` list. It runs asynchronously by default (returns `202` with a `job_id`); pass `async=false` to wait for the result (`200`).
- **Delete everything** — `POST /v2/corpora/{corpus_key}/reset` empties the corpus but keeps it (returns `204`).

Because this notebook owns its corpus, none of the other notebooks in the series depend on it — you can run it any time.

## About Vectara

[Vectara](https://vectara.com/) is an Agent Platform for trusted enterprise AI — a unified Agentic RAG platform with built-in retrieval, orchestration, and governance. See [Notebook 1](1-corpus-creation.ipynb) for the full overview of features and deployment options.

## Setup

This notebook only needs a `VECTARA_API_KEY` environment variable set to your personal API key.

In [1]:
import os
import json
import requests
from time import sleep

api_key = os.environ['VECTARA_API_KEY']
BASE_URL = "https://api.vectara.io/v2"

# Common headers (Content-Type is needed for corpus creation and indexing;
# it's harmless on the GET/DELETE/reset calls).
headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json",
}

# Dedicated corpus for this tutorial.
CORPUS_KEY = "tutorial-document-deletion"


def list_documents(metadata_filter=None):
    """Return all documents in CORPUS_KEY, paging through the list endpoint.

    Pass ``metadata_filter`` (e.g. "doc.category = 'finance'") to list only the
    documents that match — the list endpoint accepts the same filter syntax as
    the delete and query APIs.
    """
    documents = []
    page_key = None
    while True:
        params = {"limit": 100}
        if page_key:
            params["page_key"] = page_key
        if metadata_filter:
            params["metadata_filter"] = metadata_filter
        resp = requests.get(
            f"{BASE_URL}/corpora/{CORPUS_KEY}/documents",
            headers=headers,
            params=params,
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        documents.extend(data.get("documents", []))
        page_key = data.get("metadata", {}).get("page_key")
        if not page_key:
            break
    return documents


def wait_until_count(expected, metadata_filter=None, attempts=10, interval=1):
    """Re-list until the document count equals ``expected``, then return it.

    A delete can take a moment to be reflected in the list endpoint, so instead
    of guessing a fixed sleep we poll: re-list up to ``attempts`` times,
    ``interval`` seconds apart, and return as soon as the count matches (or once
    the attempts run out). Returns the final document list.
    """
    documents = list_documents(metadata_filter=metadata_filter)
    for _ in range(attempts):
        if len(documents) == expected:
            break
        sleep(interval)
        documents = list_documents(metadata_filter=metadata_filter)
    return documents

## Step 1: Create the corpus

We create a corpus with two document-level filter attributes, `category` and `year`, so we can later delete documents by metadata. Corpus creation is idempotent (a `409` means it already exists). We then reset it so the notebook starts from an empty corpus even on a re-run.

In [2]:
corpus_config = {
    "key": CORPUS_KEY,
    "name": "Document Deletion Demo",
    "description": "Self-contained corpus for the document-deletion tutorial",
    "encoder_name": "boomerang-2023-q3",
    "filter_attributes": [
        {
            "name": "category",
            "level": "document",
            "description": "Document category (e.g., news, finance)",
            "indexed": True,
            "type": "text",
        },
        {
            "name": "year",
            "level": "document",
            "description": "Publication year",
            "indexed": True,
            "type": "integer",
        },
    ],
}

resp = requests.post(f"{BASE_URL}/corpora", headers=headers, json=corpus_config, timeout=30)
if resp.status_code == 201:
    print(f"✓ Created corpus '{CORPUS_KEY}'")
elif resp.status_code == 409:
    print(f"✓ Corpus '{CORPUS_KEY}' already exists — reusing it")
else:
    raise RuntimeError(f"Error creating corpus: {resp.status_code} - {resp.text}")

# Start from a clean slate so counts are predictable on re-runs.
reset = requests.post(f"{BASE_URL}/corpora/{CORPUS_KEY}/reset", headers=headers, timeout=60)
print(f"  Reset status: {reset.status_code}")  # 204 = emptied

✓ Created corpus 'tutorial-document-deletion'


  Reset status: 204


## Step 2: Add a few documents

We index six small documents with the Core Indexing API. Four carry `category` and `year` metadata; two are left without any metadata to show that a metadata filter only ever touches the documents that actually have the matching field.

In [3]:
documents = [
    {
        "id": "press-2023-vectara",
        "type": "core",
        "document_parts": [{"text": "Vectara announced a new release in 2023."}],
        "metadata": {"category": "news", "year": 2023},
    },
    {
        "id": "press-2024-launch",
        "type": "core",
        "document_parts": [{"text": "A major product launch happened in 2024."}],
        "metadata": {"category": "news", "year": 2024},
    },
    {
        "id": "finance-2023-report",
        "type": "core",
        "document_parts": [{"text": "The 2023 annual financial report and outlook."}],
        "metadata": {"category": "finance", "year": 2023},
    },
    {
        "id": "finance-2024-earnings",
        "type": "core",
        "document_parts": [{"text": "Quarterly earnings figures for 2024."}],
        "metadata": {"category": "finance", "year": 2024},
    },
    # Two documents without any metadata.
    {
        "id": "note-alpha",
        "type": "core",
        "document_parts": [{"text": "An untagged scratch note about nothing in particular."}],
    },
    {
        "id": "note-beta",
        "type": "core",
        "document_parts": [{"text": "Another untagged note kept for reference."}],
    },
]

for doc in documents:
    resp = requests.post(
        f"{BASE_URL}/corpora/{CORPUS_KEY}/documents",
        headers=headers,
        json=doc,
        timeout=30,
    )
    ok = resp.status_code in (200, 201)
    print(f"  {'✓' if ok else '✗'} {doc['id']} ({resp.status_code})")

# A freshly-indexed document is searchable right away but can take a moment to
# show up in the list endpoint, so wait until all of them are listed.
indexed = wait_until_count(len(documents))
print(f"\nDocuments in corpus: {len(indexed)}")

  ✓ press-2023-vectara (201)


  ✓ press-2024-launch (201)


  ✓ finance-2023-report (201)


  ✓ finance-2024-earnings (201)


  ✓ note-alpha (201)


  ✓ note-beta (201)



Documents in corpus: 6


## Inspect the documents

List the documents so we can see their IDs and metadata. The document ID is what you pass to the single-document delete endpoint.

In [4]:
docs = wait_until_count(len(documents))
print(f"{len(docs)} documents in '{CORPUS_KEY}':\n")
for d in docs:
    md_ = d.get("metadata", {})
    print(f"  • id={d['id']:24} category={md_.get('category', '-'):8} year={md_.get('year', '-')}")

6 documents in 'tutorial-document-deletion':

  • id=press-2023-vectara       category=news     year=2023
  • id=note-alpha               category=-        year=-
  • id=note-beta                category=-        year=-
  • id=press-2024-launch        category=news     year=2024
  • id=finance-2023-report      category=finance  year=2023
  • id=finance-2024-earnings    category=finance  year=2024


### 1. Delete a specific document by ID

We delete `note-alpha` with the single-document delete endpoint. To remove several known IDs in one call, use the bulk endpoint with `document_ids=id1,id2,...` instead.

In [5]:
target_id = "note-alpha"
resp = requests.delete(
    f"{BASE_URL}/corpora/{CORPUS_KEY}/documents/{target_id}",
    headers=headers,
    timeout=30,
)
print(f"Delete '{target_id}' status: {resp.status_code}")  # 204 = deleted

remaining = wait_until_count(len(docs) - 1)
print(f"Documents now: {len(remaining)} (was {len(docs)})")
assert target_id not in {d['id'] for d in remaining}, "document was not deleted"
print(f"✓ '{target_id}' is gone")

Delete 'note-alpha' status: 204
Documents now: 5 (was 6)
✓ 'note-alpha' is gone


### 2. Delete documents by metadata filter

The bulk delete endpoint accepts a `metadata_filter` using the same `doc.<field>` syntax as a query filter. Here we delete every document whose `category` is `finance`. We pass `async=false` so the call waits for the deletion and returns the result instead of a `job_id` to poll.

Note that the two untagged notes have no `category` field, so the filter never matches them.

In [6]:
metadata_filter = "doc.category = 'finance'"

# Show which documents match before deleting.
matches = list_documents(metadata_filter=metadata_filter)
print(f"{len(matches)} documents match {metadata_filter}:")
for d in matches:
    print(f"  • {d['id']}")

# Bulk delete them. async=false -> wait for completion and return results (200)
# instead of returning a job_id immediately (202).
resp = requests.delete(
    f"{BASE_URL}/corpora/{CORPUS_KEY}/documents",
    headers=headers,
    params={"metadata_filter": metadata_filter, "async": "false"},
    timeout=120,
)
print(f"\nBulk delete status: {resp.status_code}")
try:
    print(json.dumps(resp.json(), indent=2))
except ValueError:
    pass  # empty body

# The list endpoint can lag a moment behind the delete, so poll until the
# filtered set is empty rather than guessing a fixed wait.
after = wait_until_count(0, metadata_filter=metadata_filter)
total = list_documents()
print(f"\nMatching '{metadata_filter}' after delete: {len(after)}")
print(f"Total documents now: {len(total)}  (the untagged notes are untouched)")

2 documents match doc.category = 'finance':
  • finance-2023-report
  • finance-2024-earnings



Bulk delete status: 200
{
  "response_type": "success",
  "job_id": "job_SDIzYktHMzNHMlJpZkzTS1pH6sgmcyW5XtEaReE/K9iMZg==",
  "cutoff_timestamp": "2026-05-28T23:28:32.552Z",
  "deleted_count": 2,
  "skipped_count": 0
}



Matching 'doc.category = 'finance'' after delete: 0
Total documents now: 3  (the untagged notes are untouched)


### 3. Delete all documents (reset the corpus)

To remove everything at once, reset the corpus. This deletes all documents and their data but keeps the corpus itself along with its configuration (encoder, filter attributes).

In [7]:
resp = requests.post(
    f"{BASE_URL}/corpora/{CORPUS_KEY}/reset",
    headers=headers,
    timeout=60,
)
print(f"Reset status: {resp.status_code}")  # 204 = corpus emptied

# Poll until the corpus reports empty (reset can take a moment to reflect).
remaining = wait_until_count(0)
print(f"Documents after reset: {len(remaining)}")
assert len(remaining) == 0, f"expected empty corpus, found {len(remaining)}"
print("✓ Corpus is empty")

Reset status: 204
Documents after reset: 0
✓ Corpus is empty


### Clean up (optional)

The corpus is now empty but still exists. If you don't need it anymore, delete the corpus itself.

In [8]:
resp = requests.delete(f"{BASE_URL}/corpora/{CORPUS_KEY}", headers=headers, timeout=30)
print(f"Delete corpus status: {resp.status_code}")  # 204 = deleted

Delete corpus status: 204
